**INSTALLAZIONE DELLE LIBRERIE**

In [1]:
!pip install -q -U transformers bitsandbytes accelerate peft

**TESTO IL MODELLO**

In [2]:
mock_input_from_retriever = {
    "sub_claim": "La vitamina D riduce il rischio di infezione da COVID-19",
    "evidence_passages": [
        {
            "id": "PUBMED_123",
            "text": "Uno studio randomizzato controllato su 5000 pazienti non ha mostrato una riduzione significativa dell'incidenza di COVID-19 con l'integrazione di Vitamina D.",
            "source": "Journal of Internal Medicine, 2022"
        },
        {
            "id": "PMC_456",
            "text": "Sebbene la carenza di Vitamina D sia associata a casi più gravi, non ci sono prove che l'integrazione prevenga l'infezione iniziale.",
            "source": "Nature Reviews, 2021"
        }
    ]
}

**inizializzazione agente**

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class ReasoningAgent:
    def __init__(self, model_id="Qwen/Qwen2.5-7B-Instruct"):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
            attn_implementation="sdpa" #ottimizzazione kv cache
            
        )

    def generate_cot(self, sub_claim, evidence_list):
        evidence_text = "\n".join([f"- {e['text']} (Fonte: {e['source']})" for e in evidence_list])

        prompt = f"""Sei un esperto di fact-checking biomedico. Analizza il seguente claim basandoti sulle evidenze fornite.
Genera un ragionamento passo-passo (Chain-of-Thought) che colleghi le prove al claim, evidenziando eventuali concordanze o contraddizioni.

CLAIM: {sub_claim}

EVIDENZE SCIENTIFICHE:
{evidence_text}

RAGIONAMENTO LOGICO:"""

        messages = [
            {"role": "system", "content": "Sei un analista scientifico rigoroso e imparziale."},
            {"role": "user", "content": prompt}
        ]

        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True
        )

        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return response.split("assistant\n")[-1].strip()

agent = ReasoningAgent()
reasoning = agent.generate_cot(mock_input_from_retriever["sub_claim"], mock_input_from_retriever["evidence_passages"])
print(reasoning)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Per analizzare il claim "La vitamina D riduce il rischio di infezione da COVID-19" basandomi sulle evidenze fornite, procederò con il ragionamento passo-passo seguendo i dati scientifici presentati:

1. **Studio Randomizzato Controllato**:
   - **Claim**: La vitamina D riduce il rischio di infezione da COVID-19.
   - **Evidenza**: Uno studio randomizzato controllato su 5000 pazienti non ha mostrato una riduzione significativa dell'incidenza di COVID-19 con l'integrazione di Vitamina D.
   - **Concordanza/Contraddizione**: Questa evidenza contraddice il claim. Il risultato del studio non ha trovato una riduzione significativa dell'incidenza di infezione da COVID-19 con l'integrazione di vitamina D, quindi non supporta il claim.

2. **Associazione con Gravi Casistiche**:
   - **Claim**: La vitamina D riduce il rischio di infezione da COVID-19.
   - **Evidenza**: Sebbene la carenza di Vitamina D sia associata a casi più gravi, non ci sono prove che l'integrazione prevenga l'infezione iniz

**VERACITY**

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

class ReasoningAndVeracityPipeline:
    def __init__(self, reasoning_model_id="Qwen/Qwen2.5-7B-Instruct", veracity_model_id="cross-encoder/nli-deberta-v3-base"):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        self.reasoning_tokenizer = AutoTokenizer.from_pretrained(reasoning_model_id)
        self.reasoning_model = AutoModelForCausalLM.from_pretrained(
            reasoning_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
            attn_implementation="sdpa"
        )

        self.veracity_classifier = pipeline(
            "text-classification",
            model=veracity_model_id,
            device=self.reasoning_model.device,
            torch_dtype=torch.float16,
            truncation=True,    #tronca l'input se supera i 512 token evitando crash
            max_length=512
        )

    def _generate_reasoning(self, claim, evidence_list):
        evidence_text = "\n".join([f"- {e['testo']} (Fonte: {e.get('url', 'N/A')})" for e in evidence_list])

        prompt = f"""Sei un esperto di fact-checking biomedico. Analizza il seguente claim basandoti sulle evidenze fornite.
Genera un ragionamento passo-passo (Chain-of-Thought) che colleghi le prove al claim. Sii conciso e logico.

CLAIM: {claim}

EVIDENZE SCIENTIFICHE:
{evidence_text}

RAGIONAMENTO LOGICO:"""

        messages = [
            {"role": "system", "content": "Sei un analista scientifico rigoroso e imparziale."},
            {"role": "user", "content": prompt}
        ]

        text = self.reasoning_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self.reasoning_tokenizer([text], return_tensors="pt").to(self.reasoning_model.device)

        with torch.no_grad():
            generated_ids = self.reasoning_model.generate(
                **model_inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True
            )

        response = self.reasoning_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        del model_inputs
        del generated_ids
        torch.cuda.empty_cache()

        return response.split("assistant\n")[-1].strip()
    

    ''' Few Shot prompting
    def _generate_reasoning(self, claim, evidence_list):
        evidence_text = "\n".join([f"- {e['testo']}" for e in evidence_list])

        # --- 1. SYSTEM PROMPT (Modalità "Detective Neutrale") ---
        system_prompt = """Sei un analista biomedico neutrale. Il tuo compito è spiegare la relazione logica tra un claim e le evidenze.
REGOLE CRITICHE:
1. MASSIMA SINTESI: Il ragionamento DEVE essere di 2-3 frasi (sotto i 150 token).
2. NEUTRALITÀ ASSOLUTA: Spiega solo come le informazioni si allineano o si scontrano. NON usare MAI parole conclusive come "smentisce", "conferma", "è falso" o "è vero".
3. NIENTE PREAMBOLI: Vai dritto all'analisi dei fatti."""

        # --- 2. FEW-SHOT EXAMPLES (Insegnano l'analisi pura senza verdetto) ---
        messages = [
            {"role": "system", "content": system_prompt},

            # ESEMPIO 1 (Analisi di una discrepanza - senza dire "Falso")
            {"role": "user", "content": "CLAIM: L'aglio cura il cancro.\nEVIDENZE SCIENTIFICHE:\n- Nessuno studio clinico dimostra l'efficacia dell'aglio contro le neoplasie.\nRAGIONAMENTO LOGICO:"},
            {"role": "assistant", "content": "Il claim postula un'azione curativa dell'aglio contro i tumori. Tuttavia, le evidenze riportano un'assenza totale di studi clinici che provino tale efficacia, indicando una netta discrepanza tra l'affermazione e i dati scientifici disponibili."},

            # ESEMPIO 2 (Analisi di un allineamento - senza dire "Vero")
            {"role": "user", "content": "CLAIM: L'aspirina abbassa la temperatura corporea.\nEVIDENZE SCIENTIFICHE:\n- L'acido acetilsalicilico agisce come antipiretico inibendo le prostaglandine.\nRAGIONAMENTO LOGICO:"},
            {"role": "assistant", "content": "Il claim descrive l'abbassamento della temperatura. Le evidenze definiscono il principio attivo come 'antipiretico', un meccanismo farmacologico che ha l'effetto diretto di contrastare la febbre e ridurre il calore corporeo. I due concetti sono semanticamente sovrapponibili."},

            # --- 3. INPUT REALE DELL'UTENTE ---
            {"role": "user", "content": f"CLAIM: {claim}\nEVIDENZE SCIENTIFICHE:\n{evidence_text}\nRAGIONAMENTO LOGICO:"}
        ]

        text = self.reasoning_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self.reasoning_tokenizer([text], return_tensors="pt").to(self.reasoning_model.device)

        with torch.no_grad():
            generated_ids = self.reasoning_model.generate(
                **model_inputs,
                max_new_tokens=200, 
                temperature=0.1,
                do_sample=True
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.reasoning_tokenizer.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

        del model_inputs
        del generated_ids
        del generated_ids_trimmed
        torch.cuda.empty_cache()

        return response.strip()
    '''

    def _assess_veracity(self, claim, evidence_list, reasoning_text):
        # Uniamo le evidenze e il ragionamento per dare tutto il contesto al classificatore
        evidence_all = " ".join([e["testo"] for e in evidence_list])

        # Creiamo un'ipotesi esplicita per il modello NLI
        # Questo costringe DeBERTa a confrontare la verità dei fatti con il claim
        context = f"Scientific Evidence: {evidence_all} Analysis: {reasoning_text}"
        hypothesis = f"Based on this, it is true that {claim}"

        # Esecuzione della classificazione
        # Usiamo il formato standard [testo, ipotesi] per i modelli Cross-Encoder
        result = self.veracity_classifier([{"text": context, "text_pair": hypothesis}])[0]

        raw_label = result["label"].lower()
        score = result["score"]

        # Mapping corretto e robusto per DeBERTa-v3-base-nli
        # LABEL_0 o 'entailment' significa che il contesto conferma l'ipotesi
        if raw_label == "entailment" or raw_label == "label_0":
            final_label = "Supported"
        # LABEL_2 o 'contradiction' significa che il contesto smentisce l'ipotesi
        elif raw_label == "contradiction" or raw_label == "label_2":
            final_label = "Refuted"
        else:
            final_label = "Not Enough Information"

        return final_label, round(score, 4)

    def process_claim(self, input_data):
        claim = input_data.get("sub_claim", "")
        evidence = input_data.get("evidence_passages", [])

        chain_of_thought = self._generate_reasoning(claim, evidence)
        verdict, confidence = self._assess_veracity(claim, evidence, chain_of_thought)

        final_output = {
            "claim": claim,
            "verdict": verdict,
            "confidence_score": confidence,
            "chain_of_thought_log": chain_of_thought,
            "supporting_evidence": evidence
        }

        return final_output

    def print_readable_report(self, verdetto_json):
        print("\n" + "="*60)
        print("  REPORT MEDFACTCHECK: ANALISI COMPLETATA")
        print("="*60)

        print(f" AFFERMAZIONE: {verdetto_json['claim']}")

        percentuale_sicurezza = verdetto_json['confidence_score'] * 100
        print(f" VERDETTO FINALE: {verdetto_json['verdict']} (Sicurezza: {percentuale_sicurezza:.1f}%)")

        print("\n RAGIONAMENTO LOGICO:")
        print(verdetto_json['chain_of_thought_log'])

        print("\n FONTI SCIENTIFICHE CONSULTATE:")
        lista_fonti = verdetto_json['supporting_evidence']

        for numero, fonte in enumerate(lista_fonti, 1):
            titolo = fonte.get('titolo', 'N/D')
            testo = fonte.get('testo', '')
            url = fonte.get('url', 'N/D')
            print(f"  {numero}. Titolo: {titolo}")
            print(f"     Testo: {testo}")
            print(f"     Link: {url}\n")

        print("="*60)


mock_input = {
    "sub_claim": "I vaccini a mRNA alterano il DNA umano.",
    "evidence_passages": [
        {
            "titolo": "Studio sui meccanismi dell'mRNA",
            "url": "https://pubmed.ncbi.nlm.nih.gov/esempio1/",
            "testo": "L'mRNA dei vaccini non entra nel nucleo della cellula dove si trova il DNA, rendendo impossibile un'alterazione del genoma umano."
        },
        {
            "titolo": "Sicurezza dei vaccini Covid-19",
            "url": "https://pubmed.ncbi.nlm.nih.gov/esempio2/",
            "testo": "Non ci sono prove cliniche o biologiche che l'RNA messaggero possa integrarsi nel DNA dell'ospite."
        }
    ]
}

my_pipeline = ReasoningAndVeracityPipeline()
risultato = my_pipeline.process_claim(mock_input)

print("\n--- JSON GREZZO PER IL DATABASE ---")
print(json.dumps(risultato, indent=2, ensure_ascii=False))

my_pipeline.print_readable_report(risultato)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (540 > 512). Running this sequence through the model will result in indexing errors



--- JSON GREZZO PER IL DATABASE ---
{
  "claim": "I vaccini a mRNA alterano il DNA umano.",
  "verdict": "Refuted",
  "confidence_score": 0.9918,
  "chain_of_thought_log": "Passo 1: Identificare il claim da verificare.\nClaim: \"I vaccini a mRNA alterano il DNA umano.\"\n\nPasso 2: Analizzare la prima evidenza fornita.\nEvidenza: \"L'mRNA dei vaccini non entra nel nucleo della cellula dove si trova il DNA, rendendo impossibile un'alterazione del genoma umano.\"\n- Questa evidenza specifica che l'mRNA dei vaccini non raggiunge il nucleo della cellula, dove si trova il DNA.\n- Pertanto, non c'è un meccanismo per l'mRNA di alterare il DNA umano all'interno della cellula.\n\nPasso 3: Analizzare la seconda evidenza fornita.\nEvidenza: \"Non ci sono prove cliniche o biologiche che l'RNA messaggero possa integrarsi nel DNA dell'ospite.\"\n- Questa evidenza sostiene che non esistono prove che l'RNA messaggero (che include l'mRNA dei vaccini) possa integrarsi nel DNA dell'ospite.\n- Ciò sugger